# Flip scenarios v2: minimum votes to change a presidential election, 1824-2024

Draft v2 of the flip-scenario notebook, built on the generic N-candidate election
dataset (`results_1824_2024.csv`) instead of the D/R/T-only `presidential_margins.csv`.

What's different from v1 (`flip_scenarios_v1.ipynb`):
- One unified dataset spans 1824-2024, instead of a separate pre-1864 pipeline.
- Any candidate can be a flip target, not just "D" or "R" -- so 1892 can target
  Weaver directly, 1912 can target Taft (the real Republican, not "TR-as-R"),
  1968 can target Wallace, etc.
- The analysis logic (`analyze_year`, `cost_to_promote`, `cost_to_dethrone`,
  `top_vote_getter`, `national_vote_totals`) is imported from
  `build_flip_results_pre1864.py`, not reimplemented inline -- this also fixes
  a real bug in v1's inline knapsack cost function (it treated a dethroned
  third-party candidate's original vote total as a static threshold rather
  than accounting for votes moving both ways, overstating the cost to flip by
  roughly 2x in cases like 1892).
- Same two cost metrics as v1 (`'votes'` and `'margin'`), same EC-split and
  NPV-margin summary lines, same per-state before/after table -- but the
  table now shows one column per candidate actually involved in the chosen
  flip, instead of fixed `D`/`R` columns.

What's still out of scope here:
- The GIF/map generation pipeline (`tools/generate_flip_gif.mjs` ->
  `docs/utils/flipScenarios.js`) drives the *live site*, which only has
  1864+ state boundaries and only renders D/R-style "classic"/"no_majority"
  flips from `docs/flip_results.csv`. So GIF commands are only emitted below
  for the years/modes the site can actually render -- 1864+ "flip to the
  actual national runner-up" and "break majority" scenarios. Pre-1864 years
  and third-party targets (Weaver, Taft, Wallace, Perot) don't have site
  support yet and are markdown-table-only here.
- This is scoped to be *correct and useful for the years actually spotlighted
  below*, not to gracefully handle every conceivable edge case in every
  election 1824-2024.


## Step 1: Load data

In [1]:
import math
import sys
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(Path.cwd()))
from build_flip_results_pre1864 import (
    analyze_year,
    cost_to_dethrone,
    cost_to_promote,
    national_vote_totals,
    top_vote_getter,
)

DATA_FILE = 'results_1824_2024.csv'


ModuleNotFoundError: No module named 'build_flip_results_pre1864'

In [2]:
def load_units(file_path):
    """year -> unit -> {'unit_evs', 'had_popular_vote', 'state', 'candidates': {name: {...}}}.
    Same shape as build_flip_results_pre1864.load_units, just parameterized on
    the file path so it can read the unified 1824-2024 dataset instead of the
    pre-1864-only one."""
    years = defaultdict(lambda: defaultdict(lambda: {'unit_evs': 0, 'had_popular_vote': False, 'candidates': {}}))
    df = pd.read_csv(file_path)
    for r in df.to_dict('records'):
        year = int(r['year'])
        u = years[year][r['unit']]
        u['unit_evs'] = int(r['unit_evs'])
        u['had_popular_vote'] = bool(r['had_popular_vote'])
        u['state'] = r['state']
        raw_votes = r['raw_votes']
        raw_votes = int(raw_votes) if pd.notna(raw_votes) else None
        u['candidates'][r['candidate']] = {
            'raw_votes': raw_votes,
            'evs_awarded': int(r['evs_awarded']),
            'on_ballot': bool(r['on_ballot']),
            'party': r['party'] if pd.notna(r['party']) else '',
        }
    return years


years = load_units(DATA_FILE)
print(f"Loaded {len(years)} years: {min(years)}-{max(years)}")


Loaded 51 years: 1824-2024


## Step 2: Precompute scenarios for every year, both metrics

Same two cost metrics as v1: `'votes'` (raw vote count) and `'margin'` (cost in
thousandths of a percent of that unit's own turnout, which tends to favor
flipping small, close units over large ones). Reuses `analyze_year` from
`build_flip_results_pre1864.py` unchanged -- it already operates on a generic
`units_for_year` dict with no D/R/T assumptions.

In [3]:
results = {}
for year in years:
    for metric in ('votes', 'margin'):
        scenarios, totals, total_evs, winner, nat_votes = analyze_year(year, years[year], metric=metric)
        results[(year, metric)] = {
            'scenarios': scenarios, 'totals': totals, 'total_evs': total_evs,
            'winner': winner, 'nat_votes': nat_votes,
        }
print(f"Analyzed {len({y for y, m in results})} years x 2 metrics.")


Analyzed 51 years x 2 metrics.


## Step 3: Render a scenario as a markdown table

Reproduces v1's "Applied flips" summary badges, EC line, and NPV margin line,
generalized from fixed `D`/`R` to an arbitrary candidate list. Unlike v1's
fixed `D | R` columns, the per-state table shows one column per candidate
actually involved in the chosen flip -- so a scenario that moves votes away
from Cleveland in some states and Harrison in others (both toward Weaver)
shows all three names, while a simple two-way race still shows just two
columns.

In [4]:
FLIP_ALIASES = {
    'winner': 'candidate_majority', 'flip_winner': 'candidate_majority', 'classic': 'candidate_majority',
    'candidate_majority': 'candidate_majority',
    'majority': 'break_majority', 'no_majority': 'break_majority', 'break_majority': 'break_majority',
}

# Multi-word surnames where "last token" would give a wrong/awkward short form.
SURNAME_OVERRIDES = {
    'Martin Van Buren': 'Van Buren',
    'Robert La Follette': 'La Follette',
}


def short_name(name):
    """Last-name-only form for compact +N% style displays (EC line, NPV line,
    per-state Margin column) -- full names stay in table headers and titles,
    where they appear once rather than repeated."""
    if name in SURNAME_OVERRIDES:
        return SURNAME_OVERRIDES[name]
    if '(' in name:
        return name
    parts = name.split()
    return parts[-1] if parts else name


def best_other_candidate(candidates, incumbent):
    """The on-ballot candidate (other than `incumbent`) with the most votes in this unit."""
    on_ballot = {
        n: c['raw_votes'] for n, c in candidates.items()
        if c['on_ballot'] and c['raw_votes'] is not None and n != incumbent
    }
    return max(on_ballot, key=on_ballot.get) if on_ballot else None


def name_margin_str(diff, name_a, name_b):
    leader = name_a if diff >= 0 else name_b
    return f"{short_name(leader)}+{abs(diff):,}"


def render_flip_scenario(year, flip_type, target=None, metric='votes', show=True):
    mode = FLIP_ALIASES.get(str(flip_type).lower())
    if mode is None:
        raise ValueError(f"Unknown flip_type {flip_type!r}; expected one of {sorted(set(FLIP_ALIASES))}")
    key = (year, metric)
    if key not in results:
        raise ValueError(f"No data for year={year}, metric={metric!r}")

    res = results[key]
    scenarios, totals, winner, nat_votes = res['scenarios'], res['totals'], res['winner'], res['nat_votes']

    if mode == 'break_majority':
        sc = next((s for s in scenarios if s['mode'] == 'break_majority'), None)
        label = f"Break {winner}'s majority"
    else:
        if target is None:
            ranked = sorted(totals.items(), key=lambda kv: -kv[1])
            target = next((c for c, ev in ranked if c != winner and c != 'Other'), None)
        sc = next((s for s in scenarios if s['mode'] == 'candidate_majority' and s['target_candidate'] == target), None)
        label = f"Flip to {target}"

    if sc is None:
        msg = f"No scenario found for {year} / {flip_type} / target={target!r}."
        if show:
            display(Markdown(f"*{msg}*"))
        return msg
    if sc['achievable'] is None:
        msg = f"N/A for {year}: nobody already has a majority to break."
        if show:
            display(Markdown(f"*{msg}*"))
        return msg
    if not sc['achievable']:
        msg = f"No feasible way to {label.lower()} in {year} (metric={metric}) -- impossible with the data available."
        if show:
            display(Markdown(f"*{msg}*"))
        return msg

    units_for_year = years[year]

    # --- per-unit from/to + running EC totals ---
    pre_ev = dict(totals)
    post_ev = dict(pre_ev)
    row_info = []
    involved = set()
    for u in sc['units_flipped']:
        cands = units_for_year[u['abbr']]['candidates']
        if mode == 'break_majority':
            from_name, to_name = winner, best_other_candidate(cands, winner)
        else:
            from_name, to_name = top_vote_getter(cands), target
        involved.update([from_name, to_name])
        post_ev[from_name] = post_ev.get(from_name, 0) - u['ev']
        if to_name:
            post_ev[to_name] = post_ev.get(to_name, 0) + u['ev']
        row_info.append({**u, 'from': from_name, 'to': to_name, 'cands': cands})
    involved.discard(None)
    columns = sorted(involved)

    # --- summary badges ---
    metric_label = 'min margin' if metric == 'margin' else 'min votes'
    votes_sum = sc['min_votes']
    states_flipped = len(sc['units_flipped'])
    pct_of_total = sc['pct_national_vote'] if sc['pct_national_vote'] != '' else 0.0

    ec_names = [n for n in set(pre_ev) | set(post_ev)
                if n != 'Other' and (pre_ev.get(n, 0) > 0 or post_ev.get(n, 0) > 0)]
    ec_names.sort(key=lambda n: -post_ev.get(n, 0))
    ec_line1 = 'EC ' + ' | '.join(f"{short_name(n)} {pre_ev.get(n, 0)}" for n in ec_names)
    ec_line2 = 'EC ' + ' | '.join(f"{short_name(n)} {post_ev.get(n, 0)}" for n in ec_names)
    ec_line = f"{ec_line1} → {ec_line2} ({sc['need']} to win)"

    # National popular vote, before and after applying the same per-unit vote
    # shifts used for the EC line -- usually barely moves the needle, but shows
    # exactly how much (or, occasionally, whether the NPV leader itself flips).
    nat_totals = national_vote_totals(units_for_year)
    post_nat_totals = dict(nat_totals)
    for r in row_info:
        post_nat_totals[r['from']] = post_nat_totals.get(r['from'], 0) - r['votes_needed']
        if r['to']:
            post_nat_totals[r['to']] = post_nat_totals.get(r['to'], 0) + r['votes_needed']

    def npv_margin_str(totals_dict):
        ranked = sorted(totals_dict.items(), key=lambda kv: -kv[1])
        if len(ranked) < 2 or not nat_votes:
            return None
        leader, leader_v = ranked[0]
        second, second_v = ranked[1]
        margin_raw = leader_v - second_v
        margin_pct = margin_raw / nat_votes * 100
        return f"{short_name(leader)}+{margin_raw:,} ({short_name(leader)}+{margin_pct:.2f}%)"

    npv_line = ''
    pre_npv = npv_margin_str(nat_totals)
    if pre_npv:
        npv_line = f"NPV MARGIN: **{pre_npv}**"
        if row_info:
            post_npv = npv_margin_str(post_nat_totals)
            npv_line += f" → **{post_npv}**"

    # Joined with explicit <br> (not markdown's trailing-two-spaces convention) so the
    # three summary lines sit tight together with no stray paragraph gap between them.
    summary_block = (
        f"VOTES CHANGED: **{votes_sum:,}**  |  % OF NATIONAL VOTE: **{pct_of_total}%**  |  "
        f"STATES FLIPPED: **{states_flipped}**<br>{ec_line}"
    )
    if npv_line:
        summary_block += f"<br>{npv_line}"

    lines = [
        f"**{year} -- {label}** (optimize: {metric_label})",
        '',
        summary_block,
        '',
    ]

    if not sc['units_flipped']:
        lines.append(f"*Already true -- no votes need to change to {label.lower()} in {year}.*")
    else:
        lines += [
            '*Applied flips:*',
            '',
            '| State | EV | ' + ' | '.join(columns) + ' | Margin | Votes moved |',
            '|---|---|' + '---|' * len(columns) + '---|---|',
        ]
        for r in sorted(row_info, key=lambda r: r['votes_needed']):
            cands = r['cands']
            cells_out = []
            before = {}
            for c in columns:
                info = cands.get(c)
                if info is None or not info['on_ballot'] or info['raw_votes'] is None:
                    cells_out.append('—')
                    before[c] = 0
                    continue
                v0 = info['raw_votes']
                before[c] = v0
                if c == r['from']:
                    v1 = v0 - r['votes_needed']
                elif c == r['to']:
                    v1 = v0 + r['votes_needed']
                else:
                    v1 = v0
                cells_out.append(f"{v0:,}<br>→ {v1:,}" if v1 != v0 else f"{v0:,}")

            margin_before = before.get(r['from'], 0) - before.get(r['to'], 0)
            margin_after = margin_before - 2 * r['votes_needed']
            margin_cell = (f"{name_margin_str(margin_before, r['from'], r['to'])}"
                            f"<br>→ {name_margin_str(margin_after, r['from'], r['to'])}")

            pct_state = round(100.0 * r['votes_needed'] / r['total_votes'], 3) if r['total_votes'] else 0.0
            lines.append(
                f"| {r['abbr']} | {r['ev']} | " + ' | '.join(cells_out) +
                f" | {margin_cell} | {r['votes_needed']:,} ({pct_state}%) |"
            )

    md_text = '\n'.join(lines)
    if show:
        display(Markdown(md_text))
    return md_text


### Example usage

In [5]:
_ = render_flip_scenario(2000, 'winner')

**2000 -- Flip to Al Gore** (optimize: min votes)

VOTES CHANGED: **269**  |  % OF NATIONAL VOTE: **0.0003%**  |  STATES FLIPPED: **1**<br>EC Gore 267 | Bush 271 → EC Gore 292 | Bush 246 (270 to win)<br>NPV MARGIN: **Gore+375,148 (Gore+0.35%)** → **Gore+375,686 (Gore+0.35%)**

*Applied flips:*

| State | EV | Al Gore | George W. Bush | Margin | Votes moved |
|---|---|---|---|---|---|
| FL | 25 | 2,912,253<br>→ 2,912,522 | 2,912,790<br>→ 2,912,521 | Bush+537<br>→ Gore+1 | 269 (0.005%) |

In [6]:
_ = render_flip_scenario(1976, 'winner', metric='margin')

**1976 -- Flip to Gerald Ford** (optimize: min margin)

VOTES CHANGED: **23,182**  |  % OF NATIONAL VOTE: **0.0283%**  |  STATES FLIPPED: **2**<br>EC Ford 241 | Carter 297 → EC Ford 277 | Carter 261 (270 to win)<br>NPV MARGIN: **Carter+1,679,206 (Carter+2.05%)** → **Carter+1,632,842 (Carter+1.99%)**

*Applied flips:*

| State | EV | Gerald Ford | Jimmy Carter | Margin | Votes moved |
|---|---|---|---|---|---|
| OH | 25 | 2,000,505<br>→ 2,006,064 | 2,011,621<br>→ 2,006,062 | Carter+11,116<br>→ Ford+2 | 5,559 (0.135%) |
| WI | 11 | 1,004,987<br>→ 1,022,610 | 1,040,232<br>→ 1,022,609 | Carter+35,245<br>→ Ford+1 | 17,623 (0.839%) |

## Step 4: Top closest elections, full 1824-2024 range

The thing v1 couldn't do: rank every election since 1824 -- not just 1864-2024 --
by how few votes it would take to flip the winner, in one query.

In [7]:
N = 15
summary_records = []
for year in sorted({y for y, m in results}):
    res = results[(year, 'votes')]
    ranked = sorted(res['totals'].items(), key=lambda kv: -kv[1])
    runner_up = next((c for c, ev in ranked if c != res['winner']), None)
    sc = next((s for s in res['scenarios']
               if s['mode'] == 'candidate_majority' and s['target_candidate'] == runner_up), None)
    if sc is None or not sc['achievable']:
        continue
    summary_records.append({
        'year': year,
        'Winner': res['winner'],
        'Runner-up': sc['target_candidate'],
        'Votes to flip': sc['min_votes'],
        '# states flipped': len(sc['units_flipped']),
        '% of national vote': sc['pct_national_vote'],
    })

topNflip = pd.DataFrame(summary_records).sort_values('% of national vote').head(N).reset_index(drop=True)
topNflip.index += 1
display(Markdown(f"## Top {N} easiest elections to flip, 1824-2024"))
topNflip


## Top 15 easiest elections to flip, 1824-2024

,year,Winner,Runner-up,Votes to flip,# states flipped,% of national vote
1,2000,George W. Bush,Al Gore,269,1,0.0003
2,1876,Rutherford B. Hayes,Samuel J. Tilden,445,1,0.0053
3,1884,Grover Cleveland,James G. Blaine,575,1,0.0057
4,1916,Woodrow Wilson,Charles Evans Hughes,1887,1,0.0102
5,1976,Jimmy Carter,Gerald Ford,9246,2,0.0113
6,1960,John F. Kennedy,Richard Nixon,11874,5,0.0172
7,2020,Joe Biden,Donald Trump,32507,4,0.0203
8,2016,Donald Trump,Hillary Clinton,38875,3,0.0281
9,2004,George W. Bush,John Kerry,46368,4,0.0374
10,1948,Harry S. Truman,Thomas E. Dewey,29294,3,0.0600


## Step 5: Top closest elections by fragility

The elections (1824-2024) where the fewest raw popular votes, moved in the
cheapest combination of states, would have flipped the winner or broken the
electoral college majority -- whichever of the two is cheaper for that year.

In [8]:
summary_records = []
for year in sorted({y for y, m in results}):
    res = results[(year, 'votes')]
    ranked = sorted(res['totals'].items(), key=lambda kv: -kv[1])
    runner_up = next((c for c, ev in ranked if c != res['winner']), None)

    candidates_ = []
    cm = next((s for s in res['scenarios']
               if s['mode'] == 'candidate_majority' and s['target_candidate'] == runner_up), None)
    if cm and cm['achievable']:
        candidates_.append(('Flip winner', cm['min_votes'], len(cm['units_flipped']), cm['pct_national_vote']))
    bm = next((s for s in res['scenarios'] if s['mode'] == 'break_majority'), None)
    if bm and bm['achievable']:
        candidates_.append(('Break majority', bm['min_votes'], len(bm['units_flipped']), bm['pct_national_vote']))
    if not candidates_:
        continue

    mode_label, cost, n_states, pct = min(candidates_, key=lambda c: c[1])
    summary_records.append({
        'year': year, 'Winner': res['winner'], 'mode': mode_label,
        'Votes to flip': cost, '# states flipped': n_states, '% of national vote': pct,
    })

topNfragile = pd.DataFrame(summary_records).sort_values('% of national vote').head(N).reset_index(drop=True)
topNfragile.index += 1
display(Markdown(f"## Top {N} most fragile elections electorally, 1824-2024"))
topNfragile


## Top 15 most fragile elections electorally, 1824-2024

,year,Winner,mode,Votes to flip,# states flipped,% of national vote
1,2000,George W. Bush,Flip winner,269,1,0.0003
2,1876,Rutherford B. Hayes,Flip winner,445,1,0.0053
3,1884,Grover Cleveland,Flip winner,575,1,0.0057
4,1960,John F. Kennedy,Break majority,6883,4,0.0100
5,1916,Woodrow Wilson,Flip winner,1887,1,0.0102
6,1976,Jimmy Carter,Flip winner,9246,2,0.0113
7,2020,Joe Biden,Break majority,21461,3,0.0134
8,2004,George W. Bush,Break majority,18776,3,0.0152
9,2016,Donald Trump,Break majority,30768,3,0.0223
10,1948,Harry S. Truman,Break majority,12487,2,0.0256


## Step 6: GIF commands (1864+, D/R-style flips only)

The live site's map only has 1864+ state boundaries and only renders the
"classic"/"no_majority" flip types already present in `docs/flip_results.csv`
-- so this only prints commands for the "flip to the actual national
runner-up" and "break majority" spotlights below, not pre-1864 years or
third-party targets (Weaver, Taft, Wallace, Perot don't have site support
yet). Requires `npm start` running in another terminal and ffmpeg on PATH;
see `tools/generate_flip_gif.mjs`'s header comment.

In [9]:
_MODE_TOKEN = {'Flip winner': 'classic', 'Break majority': 'no_majority'}
gif_year_modes = sorted({
    (row['year'], _MODE_TOKEN[row['mode']])
    for _, row in topNfragile.iterrows() if row['year'] >= 1864
} | {(row['year'], 'classic') for _, row in topNflip.iterrows() if row['year'] >= 1864})

for year, mode in gif_year_modes:
    print(f"node tools/generate_flip_gif.mjs --year {year} --mode {mode} --metric votes --dim --rendervotes")


node tools/generate_flip_gif.mjs --year 1876 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1880 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1884 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1888 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1916 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1948 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1948 --mode no_majority --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1960 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1960 --mode no_majority --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1968 --mode no_majority --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --yea

## Election focus

Spotlighted years. Pre-1864 additions (1824, 1836, 1844) picked from
`pre1864_flip_results.csv` for having among the tightest `break_majority` /
`candidate_majority` margins in the whole 1824-1860 range. No GIFs for these
three, or for the third-party-target years (1892, 1912, 1968, 1992) -- see
the note above Step 6.

### 1824

No candidate had a majority (four-way race, decided by the House) -- so
`break_majority` is N/A. Jackson needed only 3,170 votes (0.85% of the
national vote) to reach a majority outright; Adams needed 5,408 (1.45%).
Crawford needed 41,788 (11.2%). Clay is impossible -- he wasn't competitive
enough anywhere to close the gap.

In [10]:
_ = render_flip_scenario(1824, 'candidate_majority', 'Andrew Jackson')

**1824 -- Flip to Andrew Jackson** (optimize: min votes)

VOTES CHANGED: **3,170**  |  % OF NATIONAL VOTE: **0.8475%**  |  STATES FLIPPED: **2**<br>EC Jackson 99 | Adams 84 | Clay 37 | Crawford 41 → EC Jackson 139 | Adams 84 | Clay 21 | Crawford 17 (131 to win)<br>NPV MARGIN: **Jackson+28,869 (Jackson+7.72%)** → **Jackson+32,039 (Jackson+8.57%)**

*Applied flips:*

| State | EV | Andrew Jackson | Henry Clay | William Crawford | Margin | Votes moved |
|---|---|---|---|---|---|---|
| OH | 16 | 18,411<br>→ 18,789 | 19,165<br>→ 18,787 | — | Clay+754<br>→ Jackson+2 | 378 (0.752%) |
| VA | 24 | 2,975<br>→ 5,767 | 419 | 8,558<br>→ 5,766 | Crawford+5,583<br>→ Jackson+1 | 2,792 (18.053%) |

In [11]:
_ = render_flip_scenario(1824, 'candidate_majority', 'John Quincy Adams')

**1824 -- Flip to John Quincy Adams** (optimize: min votes)

VOTES CHANGED: **5,408**  |  % OF NATIONAL VOTE: **1.4459%**  |  STATES FLIPPED: **5**<br>EC Adams 84 | Jackson 99 | Clay 37 | Crawford 41 → EC Adams 133 | Jackson 77 | Clay 34 | Crawford 17 (131 to win)<br>NPV MARGIN: **Jackson+28,869 (Jackson+7.72%)** → **Jackson+21,552 (Jackson+5.76%)**

*Applied flips:*

| State | EV | Andrew Jackson | Henry Clay | John Quincy Adams | William Crawford | Margin | Votes moved |
|---|---|---|---|---|---|---|---|
| MD | 11 | 14,473<br>→ 14,331 | 695 | 14,191<br>→ 14,333 | 3,371 | Jackson+282<br>→ Adams+2 | 142 (0.434%) |
| MS | 3 | 3,313<br>→ 2,515 | 21 | 1,718<br>→ 2,516 | 121 | Jackson+1,595<br>→ Adams+1 | 798 (15.426%) |
| NJ | 8 | 10,342<br>→ 9,373 | — | 8,405<br>→ 9,374 | 1,232 | Jackson+1,937<br>→ Adams+1 | 969 (4.799%) |
| MO | 3 | 1,166 | 2,042<br>→ 1,066 | 191<br>→ 1,167 | 34 | Clay+1,851<br>→ Adams+101 | 976 (28.43%) |
| VA | 24 | 2,975 | 419 | 3,514<br>→ 6,037 | 8,558<br>→ 6,035 | Crawford+5,044<br>→ Adams+2 | 2,523 (16.313%) |

### 1836

Van Buren's majority was breakable with just 1,288 votes (0.086% of the
national vote) -- one of the tightest margins in this entire dataset, pre- or
post-1864.

In [12]:
_ = render_flip_scenario(1836, 'break_majority')

**1836 -- Break Martin Van Buren's majority** (optimize: min votes)

VOTES CHANGED: **1,288**  |  % OF NATIONAL VOTE: **0.0858%**  |  STATES FLIPPED: **5**<br>EC Van Buren 170 | Harrison 73 | White 26 | Webster 14 | Mangum 11 → EC Van Buren 146 | Harrison 85 | White 38 | Webster 14 | Mangum 11 (148 to win)<br>NPV MARGIN: **Van Buren+213,384 (Van Buren+14.22%)** → **Van Buren+211,720 (Van Buren+14.11%)**

*Applied flips:*

| State | EV | Hugh L. White | Martin Van Buren | William H. Harrison | Margin | Votes moved |
|---|---|---|---|---|---|---|
| RI | 4 | — | 2,964<br>→ 2,836 | 2,710<br>→ 2,838 | Van Buren+254<br>→ Harrison+2 | 128 (2.256%) |
| LA | 5 | 3,583<br>→ 3,713 | 3,842<br>→ 3,712 | — | Van Buren+259<br>→ White+1 | 130 (1.751%) |
| CT | 8 | — | 19,294<br>→ 19,046 | 18,799<br>→ 19,047 | Van Buren+495<br>→ Harrison+1 | 248 (0.651%) |
| MS | 4 | 9,782<br>→ 10,040 | 10,297<br>→ 10,039 | — | Van Buren+515<br>→ White+1 | 258 (1.285%) |
| AR | 3 | 1,334<br>→ 1,858 | 2,380<br>→ 1,856 | — | Van Buren+1,046<br>→ White+2 | 524 (14.109%) |

### 1844

Clay needed only 2,554 votes (0.095%) to flip Polk's win -- famously close,
and Birney's Liberty Party ran nowhere near close enough to a majority itself
(needing 673,246 votes, 24.9%) despite being a real spoiler in the Clay/Polk
race.

In [13]:
_ = render_flip_scenario(1844, 'break_majority')

**1844 -- Break James K. Polk's majority** (optimize: min votes)

VOTES CHANGED: **2,554**  |  % OF NATIONAL VOTE: **0.0945%**  |  STATES FLIPPED: **1**<br>EC Clay 105 | Polk 170 → EC Clay 141 | Polk 134 (138 to win)<br>NPV MARGIN: **Polk+39,413 (Polk+1.46%)** → **Polk+34,305 (Polk+1.27%)**

*Applied flips:*

| State | EV | Henry Clay | James K. Polk | Margin | Votes moved |
|---|---|---|---|---|---|
| NY | 36 | 232,482<br>→ 235,036 | 237,588<br>→ 235,034 | Polk+5,106<br>→ Clay+2 | 2,554 (0.526%) |

### 1876

In [14]:
_ = render_flip_scenario(1876, 'winner')

**1876 -- Flip to Samuel J. Tilden** (optimize: min votes)

VOTES CHANGED: **445**  |  % OF NATIONAL VOTE: **0.0053%**  |  STATES FLIPPED: **1**<br>EC Tilden 184 | Hayes 185 → EC Tilden 191 | Hayes 178 (185 to win)<br>NPV MARGIN: **Tilden+252,695 (Tilden+3.00%)** → **Tilden+253,585 (Tilden+3.01%)**

*Applied flips:*

| State | EV | Rutherford B. Hayes | Samuel J. Tilden | Margin | Votes moved |
|---|---|---|---|---|---|
| SC | 7 | 91,786<br>→ 91,341 | 90,897<br>→ 91,342 | Hayes+889<br>→ Tilden+1 | 445 (0.244%) |

### 1880

In [15]:
_ = render_flip_scenario(1880, 'winner')
_ = render_flip_scenario(1880, 'winner', metric='margin')

**1880 -- Flip to Winfield Scott Hancock** (optimize: min votes)

VOTES CHANGED: **7,014**  |  % OF NATIONAL VOTE: **0.0761%**  |  STATES FLIPPED: **4**<br>EC Hancock 156 | Garfield 213 → EC Hancock 185 | Garfield 184 (185 to win)<br>NPV MARGIN: **Garfield+8,744 (Garfield+0.09%)** → **Hancock+5,284 (Hancock+0.06%)**

*Applied flips:*

| State | EV | James A. Garfield | Winfield Scott Hancock | Margin | Votes moved |
|---|---|---|---|---|---|
| OR | 3 | 20,619<br>→ 20,286 | 19,955<br>→ 20,288 | Garfield+664<br>→ Hancock+2 | 333 (0.816%) |
| CT | 6 | 67,071<br>→ 65,742 | 64,415<br>→ 65,744 | Garfield+2,656<br>→ Hancock+2 | 1,329 (1.001%) |
| NH | 5 | 44,852<br>→ 42,822 | 40,794<br>→ 42,824 | Garfield+4,058<br>→ Hancock+2 | 2,030 (2.351%) |
| IN | 15 | 232,164<br>→ 228,842 | 225,522<br>→ 228,844 | Garfield+6,642<br>→ Hancock+2 | 3,322 (0.706%) |

**1880 -- Flip to Winfield Scott Hancock** (optimize: min margin)

VOTES CHANGED: **10,517**  |  % OF NATIONAL VOTE: **0.114%**  |  STATES FLIPPED: **1**<br>EC Hancock 156 | Garfield 213 → EC Hancock 191 | Garfield 178 (185 to win)<br>NPV MARGIN: **Garfield+8,744 (Garfield+0.09%)** → **Hancock+12,290 (Hancock+0.13%)**

*Applied flips:*

| State | EV | James A. Garfield | Winfield Scott Hancock | Margin | Votes moved |
|---|---|---|---|---|---|
| NY | 35 | 555,544<br>→ 545,027 | 534,511<br>→ 545,028 | Garfield+21,033<br>→ Hancock+1 | 10,517 (0.953%) |

### 1884

In [16]:
_ = render_flip_scenario(1884, 'winner')

**1884 -- Flip to James G. Blaine** (optimize: min votes)

VOTES CHANGED: **575**  |  % OF NATIONAL VOTE: **0.0057%**  |  STATES FLIPPED: **1**<br>EC Blaine 182 | Cleveland 219 → EC Blaine 218 | Cleveland 183 (201 to win)<br>NPV MARGIN: **Cleveland+57,579 (Cleveland+0.57%)** → **Cleveland+56,429 (Cleveland+0.56%)**

*Applied flips:*

| State | EV | Grover Cleveland | James G. Blaine | Margin | Votes moved |
|---|---|---|---|---|---|
| NY | 36 | 563,154<br>→ 562,579 | 562,005<br>→ 562,580 | Cleveland+1,149<br>→ Blaine+1 | 575 (0.049%) |

### 1888

In [17]:
_ = render_flip_scenario(1888, 'winner')

**1888 -- Flip to Grover Cleveland** (optimize: min votes)

VOTES CHANGED: **7,187**  |  % OF NATIONAL VOTE: **0.0631%**  |  STATES FLIPPED: **1**<br>EC Cleveland 168 | Harrison 233 → EC Cleveland 204 | Harrison 197 (201 to win)<br>NPV MARGIN: **Cleveland+94,530 (Cleveland+0.83%)** → **Cleveland+108,904 (Cleveland+0.96%)**

*Applied flips:*

| State | EV | Benjamin Harrison | Grover Cleveland | Margin | Votes moved |
|---|---|---|---|---|---|
| NY | 36 | 650,338<br>→ 643,151 | 635,965<br>→ 643,152 | Harrison+14,373<br>→ Cleveland+1 | 7,187 (0.545%) |

### 1892

New capability: target Weaver directly instead of collapsing him into a
generic "T" slate.

In [18]:
_ = render_flip_scenario(1892, 'candidate_majority', 'James Weaver')

**1892 -- Flip to James Weaver** (optimize: min votes)

VOTES CHANGED: **1,140,268**  |  % OF NATIONAL VOTE: **9.4487%**  |  STATES FLIPPED: **24**<br>EC Weaver 23 | Cleveland 271 | Harrison 150 → EC Weaver 223 | Cleveland 120 | Harrison 101 (223 to win)<br>NPV MARGIN: **Cleveland+363,099 (Cleveland+3.01%)** → **Harrison+393,297 (Harrison+3.26%)**

*Applied flips:*

| State | EV | Benjamin Harrison | Grover Cleveland | James Weaver | Margin | Votes moved |
|---|---|---|---|---|---|---|
| WY | 3 | 8,454<br>→ 8,087 | 0 | 7,722<br>→ 8,089 | Harrison+732<br>→ Weaver+2 | 367 (2.193%) |
| NE-AL | 8 | 87,213<br>→ 85,173 | 24,943 | 83,134<br>→ 85,174 | Harrison+4,079<br>→ Weaver+1 | 2,040 (1.019%) |
| OR | 4 | 35,002<br>→ 30,983 | 14,243 | 26,965<br>→ 30,984 | Harrison+8,037<br>→ Weaver+1 | 4,019 (5.12%) |
| SD | 4 | 34,888<br>→ 30,715 | 9,081 | 26,544<br>→ 30,717 | Harrison+8,344<br>→ Weaver+2 | 4,173 (5.918%) |
| MT | 3 | 18,871<br>→ 8,518 | 17,690 | 7,338<br>→ 17,691 | Harrison+11,533<br>→ Weaver+9,173 | 10,353 (23.286%) |
| WA | 4 | 36,460<br>→ 25,822 | 29,802 | 19,165<br>→ 29,803 | Harrison+17,295<br>→ Weaver+3,981 | 10,638 (12.093%) |
| FL | 4 | 0 | 30,153<br>→ 17,497 | 4,843<br>→ 17,499 | Cleveland+25,310<br>→ Weaver+2 | 12,656 (35.68%) |
| MS | 9 | 1,398 | 40,030<br>→ 25,073 | 10,118<br>→ 25,075 | Cleveland+29,912<br>→ Weaver+2 | 14,957 (28.479%) |
| VT | 4 | 37,992<br>→ 19,017 | 16,325 | 44<br>→ 19,019 | Harrison+37,948<br>→ Weaver+2 | 18,975 (34.008%) |
| RI | 4 | 26,975<br>→ 2,866 | 24,336 | 228<br>→ 24,337 | Harrison+26,747<br>→ Weaver+21,471 | 24,109 (45.321%) |
| SC | 9 | 13,345 | 54,680<br>→ 28,543 | 2,407<br>→ 28,544 | Cleveland+52,273<br>→ Weaver+1 | 26,137 (37.072%) |
| AL | 11 | 9,184 | 138,135<br>→ 111,559 | 84,984<br>→ 111,560 | Cleveland+53,151<br>→ Weaver+1 | 26,576 (11.428%) |
| AR | 8 | 47,072 | 87,834<br>→ 49,832 | 11,831<br>→ 49,833 | Cleveland+76,003<br>→ Weaver+1 | 38,002 (25.657%) |
| GA | 13 | 48,408 | 129,446<br>→ 85,692 | 41,939<br>→ 85,693 | Cleveland+87,507<br>→ Weaver+1 | 43,754 (19.61%) |
| ME-AL | 6 | 62,936<br>→ 17,282 | 48,049 | 2,396<br>→ 48,050 | Harrison+60,540<br>→ Weaver+30,768 | 45,654 (39.204%) |
| NC | 11 | 100,346 | 132,951<br>→ 76,940 | 44,336<br>→ 100,347 | Cleveland+88,615<br>→ Weaver+23,407 | 56,011 (19.985%) |
| TX | 15 | 81,144 | 239,148<br>→ 169,417 | 99,688<br>→ 169,419 | Cleveland+139,460<br>→ Weaver+2 | 69,731 (16.518%) |
| MN | 9 | 122,823<br>→ 51,215 | 100,920 | 29,313<br>→ 100,921 | Harrison+93,510<br>→ Weaver+49,706 | 71,608 (26.796%) |
| TN | 12 | 100,537 | 136,468<br>→ 59,848 | 23,918<br>→ 100,538 | Cleveland+112,550<br>→ Weaver+40,690 | 76,620 (28.834%) |
| MD | 8 | 92,736 | 113,866<br>→ 21,925 | 796<br>→ 92,737 | Cleveland+113,070<br>→ Weaver+70,812 | 91,941 (43.109%) |
| CA | 9 | 118,027 | 118,174<br>→ 25,457 | 25,311<br>→ 118,028 | Cleveland+92,863<br>→ Weaver+92,571 | 92,717 (34.389%) |
| VA | 12 | 113,098 | 164,136<br>→ 63,312 | 12,275<br>→ 113,099 | Cleveland+151,861<br>→ Weaver+49,787 | 100,824 (34.501%) |
| KY | 13 | 135,462 | 175,461<br>→ 63,498 | 23,500<br>→ 135,463 | Cleveland+151,961<br>→ Weaver+71,965 | 111,963 (32.847%) |
| MO | 17 | 227,646 | 268,400<br>→ 81,957 | 41,204<br>→ 227,647 | Cleveland+227,196<br>→ Weaver+145,690 | 186,443 (34.426%) |

### 1912

New capability: target the real Republican (Taft), not Theodore
Roosevelt-as-R.

In [19]:
_ = render_flip_scenario(1912, 'candidate_majority', 'William H. Taft')

**1912 -- Flip to William H. Taft** (optimize: min votes)

VOTES CHANGED: **514,761**  |  % OF NATIONAL VOTE: **3.4216%**  |  STATES FLIPPED: **26**<br>EC Taft 8 | Wilson 433 | Roosevelt 90 → EC Taft 266 | Wilson 190 | Roosevelt 75 (266 to win)<br>NPV MARGIN: **Wilson+2,173,563 (Wilson+14.45%)** → **Wilson+1,721,144 (Wilson+11.44%)**

*Applied flips:*

| State | EV | Theodore Roosevelt | William H. Taft | Woodrow Wilson | Margin | Votes moved |
|---|---|---|---|---|---|---|
| WY | 3 | 9,232 | 14,560<br>→ 14,936 | 15,310<br>→ 14,934 | Wilson+750<br>→ Taft+2 | 376 (0.889%) |
| ID | 4 | 25,527 | 32,810<br>→ 33,366 | 33,921<br>→ 33,365 | Wilson+1,111<br>→ Taft+1 | 556 (0.526%) |
| NH | 4 | 17,794 | 32,927<br>→ 33,826 | 34,724<br>→ 33,825 | Wilson+1,797<br>→ Taft+1 | 899 (1.022%) |
| NM | 3 | 8,347 | 17,733<br>→ 19,086 | 20,437<br>→ 19,084 | Wilson+2,704<br>→ Taft+2 | 1,353 (2.74%) |
| RI | 5 | 16,878 | 27,703<br>→ 29,058 | 30,412<br>→ 29,057 | Wilson+2,709<br>→ Taft+1 | 1,355 (1.74%) |
| NV | 3 | 5,620 | 3,196<br>→ 5,621 | 7,986<br>→ 5,561 | Wilson+4,790<br>→ Taft+60 | 2,425 (12.056%) |
| CT | 7 | 34,129 | 68,324<br>→ 71,443 | 74,561<br>→ 71,442 | Wilson+6,237<br>→ Taft+1 | 3,119 (1.638%) |
| ND | 5 | 25,726 | 23,090<br>→ 26,323 | 29,555<br>→ 26,322 | Wilson+6,465<br>→ Taft+1 | 3,233 (3.734%) |
| DE | 3 | 8,886 | 15,998<br>→ 19,315 | 22,631<br>→ 19,314 | Wilson+6,633<br>→ Taft+1 | 3,317 (6.812%) |
| AZ | 3 | 6,949 | 3,021<br>→ 6,950 | 10,324<br>→ 6,395 | Wilson+7,303<br>→ Taft+555 | 3,929 (16.563%) |
| MT | 4 | 22,456 | 18,512<br>→ 23,227 | 27,941<br>→ 23,226 | Wilson+9,429<br>→ Taft+1 | 4,715 (5.907%) |
| OR | 5 | 37,600 | 34,673<br>→ 40,869 | 47,064<br>→ 40,868 | Wilson+12,391<br>→ Taft+1 | 6,196 (4.521%) |
| MA | 18 | 142,228 | 155,948<br>→ 164,679 | 173,408<br>→ 164,677 | Wilson+17,460<br>→ Taft+2 | 8,731 (1.789%) |
| OK | 10 | 0 | 90,786<br>→ 104,972 | 119,156<br>→ 104,970 | Wilson+28,370<br>→ Taft+2 | 14,186 (5.589%) |
| FL | 6 | 4,555 | 4,279<br>→ 19,812 | 35,343<br>→ 19,810 | Wilson+31,064<br>→ Taft+2 | 15,533 (30.555%) |
| WI | 13 | 62,448 | 130,596<br>→ 147,414 | 164,230<br>→ 147,412 | Wilson+33,634<br>→ Taft+2 | 16,818 (4.205%) |
| AR | 9 | 21,644 | 25,585<br>→ 47,200 | 68,814<br>→ 47,199 | Wilson+43,229<br>→ Taft+1 | 21,615 (17.278%) |
| SC | 9 | 1,293 | 536<br>→ 24,447 | 48,357<br>→ 24,446 | Wilson+47,821<br>→ Taft+1 | 23,911 (47.49%) |
| MS | 10 | 3,549 | 1,560<br>→ 29,443 | 57,324<br>→ 29,441 | Wilson+55,764<br>→ Taft+2 | 27,883 (43.241%) |
| MI | 15 | 214,584<br>→ 183,413 | 152,244<br>→ 183,415 | 150,751 | Roosevelt+62,340<br>→ Taft+2 | 31,171 (5.657%) |
| VA | 12 | 21,776 | 23,288<br>→ 56,811 | 90,332<br>→ 56,809 | Wilson+67,044<br>→ Taft+2 | 33,523 (24.474%) |
| TN | 12 | 54,041 | 60,475<br>→ 96,749 | 133,021<br>→ 96,747 | Wilson+72,546<br>→ Taft+2 | 36,274 (14.398%) |
| AL | 12 | 22,680 | 9,807<br>→ 46,123 | 82,438<br>→ 46,122 | Wilson+72,631<br>→ Taft+1 | 36,316 (30.787%) |
| GA | 14 | 21,985 | 5,191<br>→ 49,140 | 93,087<br>→ 49,138 | Wilson+87,896<br>→ Taft+2 | 43,949 (36.181%) |
| OH | 24 | 229,807 | 278,168<br>→ 351,502 | 424,834<br>→ 351,500 | Wilson+146,666<br>→ Taft+2 | 73,334 (7.071%) |
| NY | 45 | 390,093 | 455,487<br>→ 555,531 | 655,573<br>→ 555,529 | Wilson+200,086<br>→ Taft+2 | 100,044 (6.299%) |

### 1916

In [20]:
_ = render_flip_scenario(1916, 'winner')

# Verify the "senate bonus" claim: recompute the actual 1916 electoral count using
# only each state's House-apportionment electors (unit_evs minus its 2 Senate seats).
units_1916 = years[1916]
n_units = len(units_1916)
total_ev_actual = sum(u['unit_evs'] for u in units_1916.values())
total_ev_house_only = total_ev_actual - 2 * n_units
need_actual = total_ev_actual // 2 + 1
need_house_only = total_ev_house_only // 2 + 1

winner_1916 = results[(1916, 'votes')]['winner']
ranked_1916 = sorted(results[(1916, 'votes')]['totals'].items(), key=lambda kv: -kv[1])
runner_up_1916 = next(c for c, ev in ranked_1916 if c != winner_1916)

ev_actual = defaultdict(int)
ev_house_only = defaultdict(int)
winner_states = 0
for unit in units_1916.values():
    for name, c in unit['candidates'].items():
        if c['evs_awarded'] > 0:
            ev_actual[name] += c['evs_awarded']
            ev_house_only[name] += max(0, c['evs_awarded'] - 2)
            if name == winner_1916:
                winner_states += 1

margin_house = ev_house_only[winner_1916] - need_house_only
outcome = (
    f"still clears the {need_house_only} needed for a majority by {margin_house}"
    if margin_house >= 0 else
    f"falls {abs(margin_house)} short of the {need_house_only} needed for a majority"
)

display(Markdown(
    f"| | Total EV | {winner_1916} | {runner_up_1916} | Needed | Winner |\n"
    "|---|---|---|---|---|---|\n"
    f"| Actual (House + Senate) | {total_ev_actual} | {ev_actual[winner_1916]} | {ev_actual[runner_up_1916]} | {need_actual} | "
    f"{winner_1916 if ev_actual[winner_1916] >= need_actual else runner_up_1916} |\n"
    f"| Without the Senate bonus (House only) | {total_ev_house_only} | {ev_house_only[winner_1916]} | {ev_house_only[runner_up_1916]} | {need_house_only} | "
    f"{winner_1916 if ev_house_only[winner_1916] >= need_house_only else runner_up_1916} |\n\n"
    f"{winner_1916} carried {winner_states} states, so stripping 2 EV from each state costs "
    f"{2 * winner_states} electoral votes ({ev_actual[winner_1916]} → {ev_house_only[winner_1916]}), which {outcome}. "
    f"{runner_up_1916} carried {n_units - winner_states} states, so stripping 2 EV from each state costs "
    f"{2 * (n_units - winner_states)} electoral votes ({ev_actual[runner_up_1916]} → {ev_house_only[runner_up_1916]})."
))


**1916 -- Flip to Charles Evans Hughes** (optimize: min votes)

VOTES CHANGED: **1,887**  |  % OF NATIONAL VOTE: **0.0102%**  |  STATES FLIPPED: **1**<br>EC Hughes 255 | Wilson 276 → EC Hughes 268 | Wilson 263 (266 to win)<br>NPV MARGIN: **Wilson+578,140 (Wilson+3.12%)** → **Wilson+574,366 (Wilson+3.10%)**

*Applied flips:*

| State | EV | Charles Evans Hughes | Woodrow Wilson | Margin | Votes moved |
|---|---|---|---|---|---|
| CA | 13 | 462,516<br>→ 464,403 | 466,289<br>→ 464,402 | Wilson+3,773<br>→ Hughes+1 | 1,887 (0.189%) |

| | Total EV | Woodrow Wilson | Charles Evans Hughes | Needed | Winner |
|---|---|---|---|---|---|
| Actual (House + Senate) | 531 | 276 | 255 | 266 | Woodrow Wilson |
| Without the Senate bonus (House only) | 435 | 216 | 219 | 218 | Charles Evans Hughes |

Woodrow Wilson carried 30 states, so stripping 2 EV from each state costs 60 electoral votes (276 → 216), which falls 2 short of the 218 needed for a majority. Charles Evans Hughes carried 18 states, so stripping 2 EV from each state costs 36 electoral votes (255 → 219).

### 1948

In [21]:
_ = render_flip_scenario(1948, 'winner')
_ = render_flip_scenario(1948, 'break_majority')

**1948 -- Flip to Thomas E. Dewey** (optimize: min votes)

VOTES CHANGED: **29,294**  |  % OF NATIONAL VOTE: **0.06%**  |  STATES FLIPPED: **3**<br>EC Dewey 189 | Truman 304 | Thurmond 38 → EC Dewey 267 | Truman 226 | Thurmond 38 (266 to win)<br>NPV MARGIN: **Truman+2,187,055 (Truman+4.48%)** → **Truman+2,128,467 (Truman+4.36%)**

*Applied flips:*

| State | EV | Harry S. Truman | Thomas E. Dewey | Margin | Votes moved |
|---|---|---|---|---|---|
| OH | 25 | 1,452,791<br>→ 1,449,237 | 1,445,684<br>→ 1,449,238 | Truman+7,107<br>→ Dewey+1 | 3,554 (0.121%) |
| CA | 25 | 1,913,134<br>→ 1,904,201 | 1,895,269<br>→ 1,904,202 | Truman+17,865<br>→ Dewey+1 | 8,933 (0.222%) |
| IL | 28 | 1,994,715<br>→ 1,977,908 | 1,961,103<br>→ 1,977,910 | Truman+33,612<br>→ Dewey+2 | 16,807 (0.422%) |

**1948 -- Break Harry S. Truman's majority** (optimize: min votes)

VOTES CHANGED: **12,487**  |  % OF NATIONAL VOTE: **0.0256%**  |  STATES FLIPPED: **2**<br>EC Truman 304 | Dewey 189 | Thurmond 38 → EC Truman 254 | Dewey 239 | Thurmond 38 (266 to win)<br>NPV MARGIN: **Truman+2,187,055 (Truman+4.48%)** → **Truman+2,162,081 (Truman+4.43%)**

*Applied flips:*

| State | EV | Harry S. Truman | Thomas E. Dewey | Margin | Votes moved |
|---|---|---|---|---|---|
| OH | 25 | 1,452,791<br>→ 1,449,237 | 1,445,684<br>→ 1,449,238 | Truman+7,107<br>→ Dewey+1 | 3,554 (0.121%) |
| CA | 25 | 1,913,134<br>→ 1,904,201 | 1,895,269<br>→ 1,904,202 | Truman+17,865<br>→ Dewey+1 | 8,933 (0.222%) |

### 1960

In [22]:
_ = render_flip_scenario(1960, 'winner')
_ = render_flip_scenario(1960, 'break_majority')

**1960 -- Flip to Richard Nixon** (optimize: min votes)

VOTES CHANGED: **11,874**  |  % OF NATIONAL VOTE: **0.0172%**  |  STATES FLIPPED: **5**<br>EC Nixon 220 | Kennedy 303 | Electors 14 → EC Nixon 270 | Kennedy 253 | Electors 14 (269 to win)<br>NPV MARGIN: **Kennedy+112,827 (Kennedy+0.16%)** → **Kennedy+89,079 (Kennedy+0.13%)**

*Applied flips:*

| State | EV | John F. Kennedy | Richard Nixon | Margin | Votes moved |
|---|---|---|---|---|---|
| HI | 3 | 92,410<br>→ 92,352 | 92,295<br>→ 92,353 | Kennedy+115<br>→ Nixon+1 | 58 (0.031%) |
| NM | 4 | 156,027<br>→ 154,879 | 153,733<br>→ 154,881 | Kennedy+2,294<br>→ Nixon+2 | 1,148 (0.369%) |
| NV | 3 | 54,880<br>→ 53,633 | 52,387<br>→ 53,634 | Kennedy+2,493<br>→ Nixon+1 | 1,247 (1.163%) |
| IL | 27 | 2,377,846<br>→ 2,373,416 | 2,368,988<br>→ 2,373,418 | Kennedy+8,858<br>→ Nixon+2 | 4,430 (0.093%) |
| MO | 13 | 972,201<br>→ 967,210 | 962,221<br>→ 967,212 | Kennedy+9,980<br>→ Nixon+2 | 4,991 (0.258%) |

**1960 -- Break John F. Kennedy's majority** (optimize: min votes)

VOTES CHANGED: **6,883**  |  % OF NATIONAL VOTE: **0.01%**  |  STATES FLIPPED: **4**<br>EC Kennedy 303 | Nixon 220 | Electors 14 → EC Kennedy 266 | Nixon 257 | Electors 14 (269 to win)<br>NPV MARGIN: **Kennedy+112,827 (Kennedy+0.16%)** → **Kennedy+99,061 (Kennedy+0.14%)**

*Applied flips:*

| State | EV | John F. Kennedy | Richard Nixon | Margin | Votes moved |
|---|---|---|---|---|---|
| HI | 3 | 92,410<br>→ 92,352 | 92,295<br>→ 92,353 | Kennedy+115<br>→ Nixon+1 | 58 (0.031%) |
| NM | 4 | 156,027<br>→ 154,879 | 153,733<br>→ 154,881 | Kennedy+2,294<br>→ Nixon+2 | 1,148 (0.369%) |
| NV | 3 | 54,880<br>→ 53,633 | 52,387<br>→ 53,634 | Kennedy+2,493<br>→ Nixon+1 | 1,247 (1.163%) |
| IL | 27 | 2,377,846<br>→ 2,373,416 | 2,368,988<br>→ 2,373,418 | Kennedy+8,858<br>→ Nixon+2 | 4,430 (0.093%) |

### 1968

How close was Wallace, really?

In [23]:
_ = render_flip_scenario(1968, 'candidate_majority', 'George Wallace')

**1968 -- Flip to George Wallace** (optimize: min votes)

VOTES CHANGED: **5,355,535**  |  % OF NATIONAL VOTE: **7.3163%**  |  STATES FLIPPED: **31**<br>EC Wallace 45 | Nixon 302 | Humphrey 191 → EC Wallace 270 | Nixon 145 | Humphrey 123 (270 to win)<br>NPV MARGIN: **Nixon+511,944 (Nixon+0.70%)** → **Humphrey+293,265 (Humphrey+0.40%)**

*Applied flips:*

| State | EV | George Wallace | Hubert Humphrey | Richard Nixon | Margin | Votes moved |
|---|---|---|---|---|---|---|
| SC | 8 | 215,430<br>→ 234,747 | 197,486 | 254,062<br>→ 234,745 | Nixon+38,632<br>→ Wallace+2 | 19,317 (2.896%) |
| TN | 11 | 424,792<br>→ 448,693 | 351,233 | 472,592<br>→ 448,691 | Nixon+47,800<br>→ Wallace+2 | 23,901 (1.914%) |
| AK | 3 | 10,024<br>→ 35,412 | 35,411 | 37,600<br>→ 12,212 | Nixon+27,576<br>→ Wallace+23,200 | 25,388 (30.575%) |
| WY | 3 | 11,105<br>→ 45,174 | 45,173 | 70,927<br>→ 36,858 | Nixon+59,822<br>→ Wallace+8,316 | 34,069 (26.783%) |
| NV | 3 | 20,432<br>→ 60,599 | 60,598 | 73,188<br>→ 33,021 | Nixon+52,756<br>→ Wallace+27,578 | 40,167 (26.046%) |
| DE | 3 | 28,459<br>→ 89,195 | 89,194 | 96,714<br>→ 35,978 | Nixon+68,255<br>→ Wallace+53,217 | 60,736 (28.333%) |
| ID | 4 | 36,541<br>→ 100,956 | 89,273 | 165,369<br>→ 100,954 | Nixon+128,828<br>→ Wallace+2 | 64,415 (22.122%) |
| VT | 3 | 5,104<br>→ 70,256 | 70,255 | 85,142<br>→ 19,990 | Nixon+80,038<br>→ Wallace+50,266 | 65,152 (40.366%) |
| NC | 13 | 496,188<br>→ 561,691 | 464,113 | 627,192<br>→ 561,689 | Nixon+131,004<br>→ Wallace+2 | 65,503 (4.126%) |
| ND | 4 | 14,244<br>→ 94,770 | 94,769 | 138,669<br>→ 58,143 | Nixon+124,425<br>→ Wallace+36,627 | 80,526 (32.486%) |
| HI | 4 | 3,469<br>→ 91,426 | 141,324<br>→ 53,367 | 91,425 | Humphrey+137,855<br>→ Wallace+38,059 | 87,957 (37.236%) |
| MT | 4 | 20,015<br>→ 114,118 | 114,117 | 138,835<br>→ 44,732 | Nixon+118,820<br>→ Wallace+69,386 | 94,103 (34.294%) |
| NM | 4 | 25,737<br>→ 130,082 | 130,081 | 169,692<br>→ 65,347 | Nixon+143,955<br>→ Wallace+64,735 | 104,345 (31.882%) |
| SD | 4 | 13,400<br>→ 118,024 | 118,023 | 149,841<br>→ 45,217 | Nixon+136,441<br>→ Wallace+72,807 | 104,624 (37.198%) |
| RI | 4 | 15,678<br>→ 131,099 | 246,518<br>→ 131,097 | 122,359 | Humphrey+230,840<br>→ Wallace+2 | 115,421 (29.979%) |
| NH | 4 | 11,173<br>→ 130,590 | 130,589 | 154,903<br>→ 35,486 | Nixon+143,730<br>→ Wallace+95,104 | 119,417 (40.167%) |
| AZ | 5 | 46,573<br>→ 170,515 | 170,514 | 266,721<br>→ 142,779 | Nixon+220,148<br>→ Wallace+27,736 | 123,942 (25.453%) |
| OK | 8 | 191,731<br>→ 320,715 | 301,658 | 449,697<br>→ 320,713 | Nixon+257,966<br>→ Wallace+2 | 128,984 (13.677%) |
| UT | 4 | 26,906<br>→ 156,666 | 156,665 | 238,728<br>→ 108,968 | Nixon+211,822<br>→ Wallace+47,698 | 129,760 (30.707%) |
| FL | 14 | 624,207<br>→ 755,506 | 676,794 | 886,804<br>→ 755,505 | Nixon+262,597<br>→ Wallace+1 | 131,299 (6.001%) |
| VA | 12 | 321,833<br>→ 456,077 | 442,387 | 590,319<br>→ 456,075 | Nixon+268,486<br>→ Wallace+2 | 134,244 (9.86%) |
| NE-AL | 5 | 44,904<br>→ 183,034 | 170,784 | 321,163<br>→ 183,033 | Nixon+276,259<br>→ Wallace+1 | 138,130 (25.73%) |
| ME-AL | 4 | 6,370<br>→ 169,255 | 217,312<br>→ 54,427 | 169,254 | Humphrey+210,942<br>→ Wallace+114,828 | 162,885 (41.453%) |
| KY | 9 | 193,098<br>→ 397,542 | 397,541 | 462,411<br>→ 257,967 | Nixon+269,313<br>→ Wallace+139,575 | 204,444 (19.362%) |
| KS | 7 | 88,921<br>→ 302,997 | 302,996 | 478,674<br>→ 264,598 | Nixon+389,753<br>→ Wallace+38,399 | 214,076 (24.528%) |
| WV | 7 | 72,560<br>→ 307,556 | 374,091<br>→ 139,095 | 307,555 | Humphrey+301,531<br>→ Wallace+168,461 | 234,996 (31.158%) |
| MD | 10 | 178,734<br>→ 517,996 | 538,310<br>→ 199,048 | 517,995 | Humphrey+359,576<br>→ Wallace+318,948 | 339,262 (27.47%) |
| IA | 9 | 66,422<br>→ 476,700 | 476,699 | 619,106<br>→ 208,828 | Nixon+552,684<br>→ Wallace+267,872 | 410,278 (35.129%) |
| IN | 13 | 243,108<br>→ 806,660 | 806,659 | 1,067,885<br>→ 504,333 | Nixon+824,777<br>→ Wallace+302,327 | 563,552 (26.538%) |
| TX | 25 | 584,269<br>→ 1,227,845 | 1,266,804<br>→ 623,228 | 1,227,844 | Humphrey+682,535<br>→ Wallace+604,617 | 643,576 (20.899%) |
| MA | 14 | 87,088<br>→ 778,154 | 1,469,218<br>→ 778,152 | 766,844 | Humphrey+1,382,130<br>→ Wallace+2 | 691,066 (29.637%) |

### 1976
Also verifying the "every election from 1960 to 1976 had a 20+ point popular-vote swing" claim, generalized to whichever two candidates actually led the national vote each year rather than assuming D/R.

In [24]:
def dem_rep_margin(year):
    """Signed D-R popular-vote margin (D positive), found via each candidate's `party`
    field rather than assuming who's D/R by name. Needed because a plain "leader's
    margin" is an unsigned magnitude -- subtracting two of those across years that
    flip which party is ahead (as 1972->1976 does) gives a nonsensical "swing"."""
    party_by_name = {}
    for unit in years[year].values():
        for name, c in unit['candidates'].items():
            if c['party'] and name not in party_by_name:
                party_by_name[name] = c['party']
    totals = national_vote_totals(years[year])
    d_votes = next((v for n, v in totals.items() if party_by_name.get(n) == 'Democratic'), 0)
    r_votes = next((v for n, v in totals.items() if party_by_name.get(n) == 'Republican'), 0)
    return d_votes - r_votes


swing_years = [1960, 1964, 1968, 1972, 1976]
swing_data = []
prev_pct = None
for yr in swing_years:
    total_yr = results[(yr, 'votes')]['nat_votes']
    npv_margin = dem_rep_margin(yr)
    npv_pct = npv_margin / total_yr * 100 if total_yr else 0.0
    swing_pts = None if prev_pct is None else npv_pct - prev_pct
    swing_dir = 'D' if swing_pts is not None and swing_pts >= 0 else 'R' if swing_pts is not None else ''
    swing_data.append((yr, npv_margin, npv_pct, swing_pts, swing_dir))
    prev_pct = npv_pct

swing_lines = [
    '**Checking the "20+ point popular-vote swing every election, 1960-1976" claim above** (signed D-R axis)',
    "",
    "| Year | NPV margin | NPV margin % | Swing from previous election |",
    "|---|---|---|---|",
]
for yr, npv_margin, npv_pct, swing_pts, swing_dir in swing_data:
    party = 'D' if npv_margin >= 0 else 'R'
    margin_disp = f"{party}+{abs(npv_margin):,}"
    pct_disp = f"{party}+{abs(npv_pct):.2f}%"
    swing_disp = "—" if swing_pts is None else f"{abs(swing_pts):.1f} pts towards {swing_dir}"
    swing_lines.append(f"| {yr} | {margin_disp} | {pct_disp} | {swing_disp} |")

display(Markdown('\n'.join(swing_lines)))

_ = render_flip_scenario(1976, 'winner')
_ = render_flip_scenario(1976, 'winner', metric='margin')


**Checking the "20+ point popular-vote swing every election, 1960-1976" claim above** (signed D-R axis)

| Year | NPV margin | NPV margin % | Swing from previous election |
|---|---|---|---|
| 1960 | D+112,827 | D+0.16% | — |
| 1964 | D+16,164,018 | D+22.81% | 22.7 pts towards D |
| 1968 | R+511,944 | R+0.70% | 23.5 pts towards R |
| 1972 | R+18,091,412 | R+23.15% | 22.4 pts towards R |
| 1976 | D+1,679,206 | D+2.05% | 25.2 pts towards D |

**1976 -- Flip to Gerald Ford** (optimize: min votes)

VOTES CHANGED: **9,246**  |  % OF NATIONAL VOTE: **0.0113%**  |  STATES FLIPPED: **2**<br>EC Ford 241 | Carter 297 → EC Ford 270 | Carter 268 (270 to win)<br>NPV MARGIN: **Carter+1,679,206 (Carter+2.05%)** → **Carter+1,660,714 (Carter+2.02%)**

*Applied flips:*

| State | EV | Gerald Ford | Jimmy Carter | Margin | Votes moved |
|---|---|---|---|---|---|
| HI | 4 | 140,003<br>→ 143,690 | 147,375<br>→ 143,688 | Carter+7,372<br>→ Ford+2 | 3,687 (1.266%) |
| OH | 25 | 2,000,505<br>→ 2,006,064 | 2,011,621<br>→ 2,006,062 | Carter+11,116<br>→ Ford+2 | 5,559 (0.135%) |

**1976 -- Flip to Gerald Ford** (optimize: min margin)

VOTES CHANGED: **23,182**  |  % OF NATIONAL VOTE: **0.0283%**  |  STATES FLIPPED: **2**<br>EC Ford 241 | Carter 297 → EC Ford 277 | Carter 261 (270 to win)<br>NPV MARGIN: **Carter+1,679,206 (Carter+2.05%)** → **Carter+1,632,842 (Carter+1.99%)**

*Applied flips:*

| State | EV | Gerald Ford | Jimmy Carter | Margin | Votes moved |
|---|---|---|---|---|---|
| OH | 25 | 2,000,505<br>→ 2,006,064 | 2,011,621<br>→ 2,006,062 | Carter+11,116<br>→ Ford+2 | 5,559 (0.135%) |
| WI | 11 | 1,004,987<br>→ 1,022,610 | 1,040,232<br>→ 1,022,609 | Carter+35,245<br>→ Ford+1 | 17,623 (0.839%) |

### 1992

And Perot?

In [25]:
_ = render_flip_scenario(1992, 'candidate_majority', 'Ross Perot')

**1992 -- Flip to Ross Perot** (optimize: min votes)

VOTES CHANGED: **6,482,749**  |  % OF NATIONAL VOTE: **6.1252%**  |  STATES FLIPPED: **36**<br>EC Perot 0 | Clinton 370 | Bush 168 → EC Perot 270 | Clinton 189 | Bush 79 (270 to win)<br>NPV MARGIN: **Clinton+5,735,450 (Clinton+5.42%)** → **Clinton+3,258,027 (Clinton+3.08%)**

*Applied flips:*

| State | EV | Bill Clinton | George H.W. Bush | Ross Perot | Margin | Votes moved |
|---|---|---|---|---|---|---|
| ME-02 | 1 | 118,229<br>→ 111,110 | 90,807 | 103,992<br>→ 111,111 | Clinton+14,237<br>→ Perot+1 | 7,119 (2.274%) |
| AK | 3 | 78,294 | 102,000<br>→ 87,740 | 73,481<br>→ 87,741 | Bush+28,519<br>→ Perot+1 | 14,260 (5.516%) |
| WY | 3 | 68,160 | 79,347<br>→ 62,449 | 51,263<br>→ 68,161 | Bush+28,084<br>→ Perot+5,712 | 16,898 (8.424%) |
| ME-01 | 1 | 145,191<br>→ 124,009 | 115,697 | 102,828<br>→ 124,010 | Clinton+42,363<br>→ Perot+1 | 21,182 (5.824%) |
| NE-01 | 1 | 80,696 | 107,081<br>→ 83,527 | 59,974<br>→ 83,528 | Bush+47,107<br>→ Perot+1 | 23,554 (9.507%) |
| NE-03 | 1 | 57,467 | 121,342<br>→ 93,407 | 65,473<br>→ 93,408 | Bush+55,869<br>→ Perot+1 | 27,935 (11.436%) |
| ME-AL | 2 | 263,420<br>→ 235,119 | 206,504 | 206,820<br>→ 235,121 | Clinton+56,600<br>→ Perot+2 | 28,301 (4.165%) |
| ND | 3 | 99,168 | 136,244<br>→ 103,663 | 71,084<br>→ 103,665 | Bush+65,160<br>→ Perot+2 | 32,581 (10.574%) |
| VT | 3 | 133,592<br>→ 99,791 | 88,122 | 65,991<br>→ 99,792 | Clinton+67,601<br>→ Perot+1 | 33,801 (11.668%) |
| ID | 4 | 137,013 | 202,645<br>→ 166,519 | 130,395<br>→ 166,521 | Bush+72,250<br>→ Perot+2 | 36,126 (7.493%) |
| MT | 3 | 154,507<br>→ 117,524 | 144,207 | 107,225<br>→ 144,208 | Clinton+47,282<br>→ Perot+26,684 | 36,983 (9.007%) |
| DE | 3 | 126,055<br>→ 82,954 | 102,313 | 59,213<br>→ 102,314 | Clinton+66,842<br>→ Perot+19,360 | 43,101 (14.876%) |
| NV | 4 | 189,148<br>→ 145,899 | 175,828 | 132,580<br>→ 175,829 | Clinton+56,568<br>→ Perot+29,930 | 43,249 (8.542%) |
| SD | 3 | 124,888 | 136,718<br>→ 85,124 | 73,295<br>→ 124,889 | Bush+63,423<br>→ Perot+39,765 | 51,594 (15.344%) |
| RI | 4 | 213,302<br>→ 159,173 | 131,601 | 105,045<br>→ 159,174 | Clinton+108,257<br>→ Perot+1 | 54,129 (11.936%) |
| UT | 5 | 183,429 | 322,632<br>→ 263,015 | 203,400<br>→ 263,017 | Bush+119,232<br>→ Perot+2 | 59,617 (8.013%) |
| KS | 6 | 390,434 | 449,951<br>→ 371,874 | 312,358<br>→ 390,435 | Bush+137,593<br>→ Perot+18,561 | 78,077 (6.747%) |
| NH | 4 | 209,040<br>→ 127,892 | 202,484 | 121,337<br>→ 202,485 | Clinton+87,703<br>→ Perot+74,593 | 81,148 (15.085%) |
| HI | 4 | 179,310<br>→ 95,490 | 136,822 | 53,003<br>→ 136,823 | Clinton+126,307<br>→ Perot+41,333 | 83,820 (22.481%) |
| DC | 3 | 192,619<br>→ 101,149 | 20,698 | 9,681<br>→ 101,151 | Clinton+182,938<br>→ Perot+2 | 91,470 (40.194%) |
| NM | 5 | 261,617<br>→ 140,687 | 212,824 | 91,895<br>→ 212,825 | Clinton+169,722<br>→ Perot+72,138 | 120,930 (21.216%) |
| WV | 5 | 331,001<br>→ 197,855 | 241,974 | 108,829<br>→ 241,975 | Clinton+222,172<br>→ Perot+44,120 | 133,146 (19.475%) |
| OR | 7 | 621,314<br>→ 487,702 | 475,757 | 354,091<br>→ 487,703 | Clinton+267,223<br>→ Perot+1 | 133,612 (9.135%) |
| OK | 8 | 473,066 | 592,929<br>→ 439,740 | 319,878<br>→ 473,067 | Bush+273,051<br>→ Perot+33,327 | 153,189 (11.018%) |
| AZ | 8 | 543,050 | 572,086<br>→ 382,776 | 353,741<br>→ 543,051 | Bush+218,345<br>→ Perot+160,275 | 189,310 (12.731%) |
| CO | 8 | 629,681<br>→ 432,840 | 562,850 | 366,010<br>→ 562,851 | Clinton+263,671<br>→ Perot+130,011 | 196,841 (12.544%) |
| WA | 11 | 993,039<br>→ 767,409 | 731,235 | 541,780<br>→ 767,410 | Clinton+451,259<br>→ Perot+1 | 225,630 (9.86%) |
| MN | 10 | 1,020,997<br>→ 791,751 | 747,841 | 562,506<br>→ 791,752 | Clinton+458,491<br>→ Perot+1 | 229,246 (9.764%) |
| CT | 8 | 682,318<br>→ 452,775 | 578,313 | 348,771<br>→ 578,314 | Clinton+333,547<br>→ Perot+125,539 | 229,543 (14.201%) |
| IA | 7 | 586,353<br>→ 334,929 | 504,891 | 253,468<br>→ 504,892 | Clinton+332,885<br>→ Perot+169,963 | 251,424 (18.561%) |
| MO | 11 | 1,053,873<br>→ 761,454 | 811,159 | 518,741<br>→ 811,160 | Clinton+535,132<br>→ Perot+49,706 | 292,419 (12.227%) |
| MA | 12 | 1,318,639<br>→ 974,684 | 805,039 | 630,731<br>→ 974,686 | Clinton+687,908<br>→ Perot+2 | 343,955 (12.401%) |
| WI | 11 | 1,041,066<br>→ 654,689 | 930,855 | 544,479<br>→ 930,856 | Clinton+496,587<br>→ Perot+276,167 | 386,377 (15.265%) |
| IN | 12 | 848,420 | 989,375<br>→ 596,888 | 455,934<br>→ 848,421 | Bush+533,441<br>→ Perot+251,533 | 392,487 (17.021%) |
| TX | 32 | 2,281,815 | 2,496,071<br>→ 1,569,036 | 1,354,781<br>→ 2,281,816 | Bush+1,141,290<br>→ Perot+712,780 | 927,035 (15.064%) |
| CA | 54 | 5,121,325<br>→ 3,708,665 | 3,630,574 | 2,296,006<br>→ 3,708,666 | Clinton+2,825,319<br>→ Perot+1 | 1,412,660 (12.69%) |

### 2000

In [26]:
_ = render_flip_scenario(2000, 'winner')

**2000 -- Flip to Al Gore** (optimize: min votes)

VOTES CHANGED: **269**  |  % OF NATIONAL VOTE: **0.0003%**  |  STATES FLIPPED: **1**<br>EC Gore 267 | Bush 271 → EC Gore 292 | Bush 246 (270 to win)<br>NPV MARGIN: **Gore+375,148 (Gore+0.35%)** → **Gore+375,686 (Gore+0.35%)**

*Applied flips:*

| State | EV | Al Gore | George W. Bush | Margin | Votes moved |
|---|---|---|---|---|---|
| FL | 25 | 2,912,253<br>→ 2,912,522 | 2,912,790<br>→ 2,912,521 | Bush+537<br>→ Gore+1 | 269 (0.005%) |

### 2004

In [27]:
_ = render_flip_scenario(2004, 'winner')
_ = render_flip_scenario(2004, 'break_majority')
_ = render_flip_scenario(2004, 'winner', metric='margin')

**2004 -- Flip to John Kerry** (optimize: min votes)

VOTES CHANGED: **46,368**  |  % OF NATIONAL VOTE: **0.0374%**  |  STATES FLIPPED: **4**<br>EC Kerry 252 | Bush 286 → EC Kerry 270 | Bush 268 (270 to win)<br>NPV MARGIN: **Bush+3,204,011 (Bush+2.59%)** → **Bush+3,111,275 (Bush+2.51%)**

*Applied flips:*

| State | EV | George W. Bush | John Kerry | Margin | Votes moved |
|---|---|---|---|---|---|
| NM | 5 | 376,930<br>→ 373,935 | 370,942<br>→ 373,937 | Bush+5,988<br>→ Kerry+2 | 2,995 (0.396%) |
| IA | 7 | 751,957<br>→ 746,927 | 741,898<br>→ 746,928 | Bush+10,059<br>→ Kerry+1 | 5,030 (0.334%) |
| NV | 5 | 418,690<br>→ 407,939 | 397,190<br>→ 407,941 | Bush+21,500<br>→ Kerry+2 | 10,751 (1.296%) |
| NE-02 | 1 | 153,041<br>→ 125,449 | 97,858<br>→ 125,450 | Bush+55,183<br>→ Kerry+1 | 27,592 (10.862%) |

**2004 -- Break George W. Bush's majority** (optimize: min votes)

VOTES CHANGED: **18,776**  |  % OF NATIONAL VOTE: **0.0152%**  |  STATES FLIPPED: **3**<br>EC Bush 286 | Kerry 252 → EC Bush 269 | Kerry 269 (270 to win)<br>NPV MARGIN: **Bush+3,204,011 (Bush+2.59%)** → **Bush+3,166,459 (Bush+2.56%)**

*Applied flips:*

| State | EV | George W. Bush | John Kerry | Margin | Votes moved |
|---|---|---|---|---|---|
| NM | 5 | 376,930<br>→ 373,935 | 370,942<br>→ 373,937 | Bush+5,988<br>→ Kerry+2 | 2,995 (0.396%) |
| IA | 7 | 751,957<br>→ 746,927 | 741,898<br>→ 746,928 | Bush+10,059<br>→ Kerry+1 | 5,030 (0.334%) |
| NV | 5 | 418,690<br>→ 407,939 | 397,190<br>→ 407,941 | Bush+21,500<br>→ Kerry+2 | 10,751 (1.296%) |

**2004 -- Flip to John Kerry** (optimize: min margin)

VOTES CHANGED: **59,301**  |  % OF NATIONAL VOTE: **0.0479%**  |  STATES FLIPPED: **1**<br>EC Kerry 252 | Bush 286 → EC Kerry 272 | Bush 266 (270 to win)<br>NPV MARGIN: **Bush+3,204,011 (Bush+2.59%)** → **Bush+3,085,409 (Bush+2.49%)**

*Applied flips:*

| State | EV | George W. Bush | John Kerry | Margin | Votes moved |
|---|---|---|---|---|---|
| OH | 20 | 2,859,768<br>→ 2,800,467 | 2,741,167<br>→ 2,800,468 | Bush+118,601<br>→ Kerry+1 | 59,301 (1.054%) |

### 2016

In [28]:
_ = render_flip_scenario(2016, 'winner')
_ = render_flip_scenario(2016, 'break_majority')

**2016 -- Flip to Hillary Clinton** (optimize: min votes)

VOTES CHANGED: **38,875**  |  % OF NATIONAL VOTE: **0.0281%**  |  STATES FLIPPED: **3**<br>EC Clinton 232 | Trump 306 → EC Clinton 278 | Trump 260 (270 to win)<br>NPV MARGIN: **Clinton+2,679,254 (Clinton+1.94%)** → **Clinton+2,757,004 (Clinton+1.99%)**

*Applied flips:*

| State | EV | Donald Trump | Hillary Clinton | Margin | Votes moved |
|---|---|---|---|---|---|
| MI | 16 | 2,279,543<br>→ 2,274,190 | 2,268,839<br>→ 2,274,192 | Trump+10,704<br>→ Clinton+2 | 5,353 (0.112%) |
| WI | 10 | 1,405,284<br>→ 1,393,909 | 1,382,536<br>→ 1,393,911 | Trump+22,748<br>→ Clinton+2 | 11,375 (0.382%) |
| PA | 20 | 2,970,733<br>→ 2,948,586 | 2,926,441<br>→ 2,948,588 | Trump+44,292<br>→ Clinton+2 | 22,147 (0.359%) |

**2016 -- Break Donald Trump's majority** (optimize: min votes)

VOTES CHANGED: **30,768**  |  % OF NATIONAL VOTE: **0.0223%**  |  STATES FLIPPED: **3**<br>EC Clinton 232 | Trump 306 → EC Clinton 269 | Trump 269 (270 to win)<br>NPV MARGIN: **Clinton+2,679,254 (Clinton+1.94%)** → **Clinton+2,740,790 (Clinton+1.98%)**

*Applied flips:*

| State | EV | Donald Trump | Hillary Clinton | Margin | Votes moved |
|---|---|---|---|---|---|
| NE-02 | 1 | 137,564<br>→ 134,296 | 131,030<br>→ 134,298 | Trump+6,534<br>→ Clinton+2 | 3,268 (1.12%) |
| MI | 16 | 2,279,543<br>→ 2,274,190 | 2,268,839<br>→ 2,274,192 | Trump+10,704<br>→ Clinton+2 | 5,353 (0.112%) |
| PA | 20 | 2,970,733<br>→ 2,948,586 | 2,926,441<br>→ 2,948,588 | Trump+44,292<br>→ Clinton+2 | 22,147 (0.359%) |

### 2020

In [29]:
_ = render_flip_scenario(2020, 'winner')
_ = render_flip_scenario(2020, 'break_majority')

**2020 -- Flip to Donald Trump** (optimize: min votes)

VOTES CHANGED: **32,507**  |  % OF NATIONAL VOTE: **0.0203%**  |  STATES FLIPPED: **4**<br>EC Trump 232 | Biden 306 → EC Trump 270 | Biden 268 (270 to win)<br>NPV MARGIN: **Biden+6,951,598 (Biden+4.34%)** → **Biden+6,886,584 (Biden+4.30%)**

*Applied flips:*

| State | EV | Donald Trump | Joe Biden | Margin | Votes moved |
|---|---|---|---|---|---|
| AZ | 11 | 1,661,686<br>→ 1,666,915 | 1,672,143<br>→ 1,666,914 | Biden+10,457<br>→ Trump+1 | 5,229 (0.154%) |
| GA | 16 | 2,461,854<br>→ 2,467,744 | 2,473,633<br>→ 2,467,743 | Biden+11,779<br>→ Trump+1 | 5,890 (0.118%) |
| WI | 10 | 1,610,184<br>→ 1,620,526 | 1,630,866<br>→ 1,620,524 | Biden+20,682<br>→ Trump+2 | 10,342 (0.314%) |
| NE-02 | 1 | 154,377<br>→ 165,423 | 176,468<br>→ 165,422 | Biden+22,091<br>→ Trump+1 | 11,046 (3.252%) |

**2020 -- Break Joe Biden's majority** (optimize: min votes)

VOTES CHANGED: **21,461**  |  % OF NATIONAL VOTE: **0.0134%**  |  STATES FLIPPED: **3**<br>EC Biden 306 | Trump 232 → EC Biden 269 | Trump 269 (270 to win)<br>NPV MARGIN: **Biden+6,951,598 (Biden+4.34%)** → **Biden+6,908,676 (Biden+4.31%)**

*Applied flips:*

| State | EV | Donald Trump | Joe Biden | Margin | Votes moved |
|---|---|---|---|---|---|
| AZ | 11 | 1,661,686<br>→ 1,666,915 | 1,672,143<br>→ 1,666,914 | Biden+10,457<br>→ Trump+1 | 5,229 (0.154%) |
| GA | 16 | 2,461,854<br>→ 2,467,744 | 2,473,633<br>→ 2,467,743 | Biden+11,779<br>→ Trump+1 | 5,890 (0.118%) |
| WI | 10 | 1,610,184<br>→ 1,620,526 | 1,630,866<br>→ 1,620,524 | Biden+20,682<br>→ Trump+2 | 10,342 (0.314%) |

### 2024

In [30]:
_ = render_flip_scenario(2024, 'winner')

**2024 -- Flip to Kamala Harris** (optimize: min votes)

VOTES CHANGED: **114,885**  |  % OF NATIONAL VOTE: **0.0732%**  |  STATES FLIPPED: **3**<br>EC Harris 226 | Trump 312 → EC Harris 270 | Trump 268 (270 to win)<br>NPV MARGIN: **Trump+2,421,484 (Trump+1.54%)** → **Trump+2,191,714 (Trump+1.40%)**

*Applied flips:*

| State | EV | Donald Trump | Kamala Harris | Margin | Votes moved |
|---|---|---|---|---|---|
| WI | 10 | 1,697,626<br>→ 1,682,927 | 1,668,229<br>→ 1,682,928 | Trump+29,397<br>→ Harris+1 | 14,699 (0.429%) |
| MI | 15 | 2,816,636<br>→ 2,776,584 | 2,736,533<br>→ 2,776,585 | Trump+80,103<br>→ Harris+1 | 40,052 (0.707%) |
| PA | 19 | 3,543,308<br>→ 3,483,174 | 3,423,042<br>→ 3,483,176 | Trump+120,266<br>→ Harris+2 | 60,134 (0.852%) |

## Notes / caveats to carry forward

- **Not yet ported from v1:** the "closeness rating" and candidate-portrait
  images seen on the live blog post are added when composing the post itself
  -- they're not generated by either notebook.
- **`results_1824_2024.csv`** is a plain concatenation of `pre1864_results.csv`
  and `results_1864_2024.csv` (see `build_1824_2024_dataset.py`) -- it is
  itself generated, not hand-edited; re-run that script after regenerating
  either source file.
- **`analyze_year`'s `metric` parameter** (`'votes'`/`'margin'`) was added to
  `build_flip_results_pre1864.py` in support of this notebook -- verified
  backward-compatible: regenerating `pre1864_flip_results.csv` and
  `flip_results_1824_2024.csv` with the change produces byte-identical output
  to before, since the default `metric='votes'` path is unchanged.
- **The live site's map/GIF pipeline is untouched** and still reads from the
  original D/R/T `docs/flip_results.csv` -- this notebook does not feed it.
